# 05a — 特征工程与模型可解释性：特征选择与 SHAP 归因分析

在第二讲我们学会了用十来个 RDKit 描述符训练随机森林预测水溶解度，在第四讲我们又用高斯过程把"模型有多大把握"量化了出来。但在真实课题汇报里，导师和审稿人最常追问的往往不是 $R^2$ 的第三位小数，而是另外两个问题：

1. **这 30 个描述符里，到底哪几个是真正起作用的？**——很多描述符彼此高度共线（`MolWt`、`HeavyAtomCount`、`MolMR`、`LabuteASA` 本质上都在说“分子有多大”），冗余特征会稀释重要性、拖慢训练、让模型在小样本上更容易过拟合。
2. **对于我合成的这一个具体分子，模型为什么判它难溶？**——全局特征重要性只能告诉你“`MolLogP` 在整个数据集上最重要”，却无法解释“为什么模型认为**蒽（anthracene）**这一个分子特别难溶”。

本节我们沿着这两条线展开：
- 前半程讲 **特征选择（Feature Selection）** 的三条经典路线——**过滤法（Filter）**、**包裹法（Wrapper）** 与 **嵌入法（Embedded）**，并亲手把 29 维描述符矩阵压到 10 维而几乎不损失精度；
- 后半程讲 **SHAP（SHapley Additive exPlanations）**：从合作博弈论的 Shapley 值定义出发，先用 16 个重训模型手写**精确 Shapley 值**验证其可加性公理，再用 `shap.TreeExplainer` 对随机森林做全局与单分子归因，最后对比 SHAP、MDI 与置换重要性在共线特征下的不同表现。

---

## 0. 环境准备与数据读取辅助函数

本节使用第一讲就开始贯穿全课程的 **AqSolDB** 水溶解度数据集（`data/solubility.csv`，9,482 个有机分子）。它自带 17 个 RDKit 描述符列，我们再用 RDKit 补算 12 个，凑成一张 **29 列的描述符矩阵**——正好是真实课题里“把能算的描述符一股脑先算出来”的典型起点。

In [ ]:
import os
import itertools
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, RFECV
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Draw, rdMolDescriptors

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110
np.random.seed(42)

def find_file(name):
    for path in (f"data/{name}", f"../data/{name}", name):
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        print(f"请上传 {name}")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"找不到 {name}，请确保它位于 data/ 文件夹或当前目录下")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 1. “能算的描述符”与“该用的描述符”

RDKit 的 `Descriptors.descList` 里有 **200 多个** 2D 描述符，`mordred` 之类的第三方库更能一口气算出 1,800 个。初学者最容易犯的错误，就是把它们**全部**扔给模型，然后祈祷算法自己分辨好坏。这会带来三个具体的麻烦：

| 问题 | 具体表现 | 在材料/化学数据上的典型例子 |
|---|---|---|
| **维数灾难与过拟合** | 特征数接近甚至超过样本数时，模型总能找到一组“碰巧”拟合训练集的权重 | 只有 40 个实测器件的新体系，却套用 1,800 维 mordred 描述符 |
| **共线性（Multicollinearity）** | 多个特征携带几乎相同的信息，重要性被稀释、线性模型系数剧烈震荡 | `MolWt` / `HeavyAtomCount` / `MolMR` / `LabuteASA` / `NumValenceElectrons` 都在描述“分子多大” |
| **准常量特征（Near-constant）** | 某一列 98% 的取值都相同，几乎不携带区分信息，却照样参与分裂 | `NumRadicalElectrons`——绝大多数稳定有机分子都是 0 |

特征选择的目标，就是在**不牺牲（甚至略微提升）预测精度**的前提下，把特征数降到可解释、可复现、可向实验同行讲清楚的规模。三条经典路线：

- **过滤法（Filter）**：只看特征矩阵 $X$ 本身（方差、相关系数）或 $X$ 与 $y$ 的单变量统计量，**与具体模型无关**，极快，适合先粗筛；
- **包裹法（Wrapper）**：把模型当黑箱反复训练，按“删掉哪一个特征掉分最少”逐步剔除（如 RFE），**最贴合最终模型**，但计算昂贵；
- **嵌入法（Embedded）**：模型训练过程本身就在做选择，如 Lasso 的 $L_1$ 稀疏化、树模型的分裂重要性。

---

## 2. 构建 29 列描述符矩阵

下面我们：
1. 读入 AqSolDB，剔除分子量过小/过大的离群条目（金属盐、聚合物片段等），随机抽取 3,000 个分子作为本节的工作集（保证 SHAP 计算在课堂时间内跑得完）；
2. 保留 CSV 自带的 17 个描述符列；
3. 用 RDKit 再补算 12 个描述符——其中刻意包含了几个**在这个数据集上几乎恒定**的列（如 `NumRadicalElectrons`），正好用来演示方差过滤。

In [ ]:
df_all = pd.read_csv(find_file("solubility.csv"))
df_all = df_all[(df_all["MolWt"] > 20) & (df_all["MolWt"] < 800)].reset_index(drop=True)
sub = df_all.sample(3000, random_state=42).reset_index(drop=True)

EXTRA_DESC = {
    "FractionCSP3":             Descriptors.FractionCSP3,
    "NHOHCount":                Descriptors.NHOHCount,
    "NOCount":                  Descriptors.NOCount,
    "HallKierAlpha":            Descriptors.HallKierAlpha,
    "Kappa1":                   Descriptors.Kappa1,
    "Kappa2":                   Descriptors.Kappa2,
    "Chi0v":                    Descriptors.Chi0v,
    "Chi1v":                    Descriptors.Chi1v,
    "NumRadicalElectrons":      Descriptors.NumRadicalElectrons,
    "NumSaturatedHeterocycles": Descriptors.NumSaturatedHeterocycles,
    "NumAmideBonds":            rdMolDescriptors.CalcNumAmideBonds,
    "MaxPartialCharge":         Descriptors.MaxPartialCharge,
}

rows, keep_idx = [], []
for i, smi in enumerate(sub["SMILES"]):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    try:
        rows.append({k: fn(mol) for k, fn in EXTRA_DESC.items()})
    except Exception:
        continue
    keep_idx.append(i)

sub = sub.loc[keep_idx].reset_index(drop=True)
X_extra = pd.DataFrame(rows).reset_index(drop=True)

BASE_COLS = [c for c in df_all.columns if c not in ("Name", "SMILES", "Solubility")]
X_full = pd.concat([sub[BASE_COLS], X_extra], axis=1)
X_full = X_full.fillna(X_full.median())        # MaxPartialCharge 在少数分子上会返回 NaN
y = sub["Solubility"].values

print(f"描述符矩阵 X_full: {X_full.shape[0]} 个分子 × {X_full.shape[1]} 个描述符")
print("列名:", list(X_full.columns))

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y, test_size=0.2, random_state=42)

rf_full = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
rf_full.fit(X_tr, y_tr)
r2_full = rf_full.score(X_te, y_te)
print(f"基准（全部 {X_full.shape[1]} 个描述符）测试集 R2 = {r2_full:.4f}")

---

## 3. 过滤法（一）：方差过滤与“准常量特征”

最朴素的过滤思路是：**一个几乎不变化的特征，不可能解释因变量的变化**。`sklearn.feature_selection.VarianceThreshold` 正是据此剔除方差低于阈值的列。

但直接对原始描述符算方差会踩一个大坑：**方差是有量纲的**。`BertzCT` 的数值范围在 $0 \sim 20{,}000$，方差自然巨大；`FractionCSP3` 被定义在 $[0, 1]$ 之间，方差再大也超不过 0.25。于是“方差排序”会退化成“量纲排序”，毫无意义。

常见的两种修正：
1. **先做 Min-Max 归一化再算方差**——把所有特征压到 $[0, 1]$ 后再比较；
2. **直接统计“最高频取值的占比”**——对 `NumRadicalElectrons` 这类整数计数型描述符更直观、更抗离群值。

下面我们把两种判据并排算出来对比：

In [ ]:
# 判据 1：Min-Max 归一化后的方差（在训练集上 fit，避免信息泄漏）
mm = MinMaxScaler().fit(X_tr)
var_scaled = pd.Series(mm.transform(X_tr).var(axis=0), index=X_tr.columns)

# 判据 2：最高频取值所占比例（越接近 1，说明该列越“恒定”）
top_share = X_tr.apply(lambda s: s.value_counts(normalize=True).iloc[0])

diag = pd.DataFrame({
    "MinMax_variance": var_scaled.round(5),
    "top_value_share": top_share.round(3),
}).sort_values("top_value_share", ascending=False)

print(diag.head(10).to_string())
print(f"
对照：MolLogP 的 MinMax 方差 = {var_scaled['MolLogP']:.5f}"
      f"，最高频取值占比 = {top_share['MolLogP']:.3f}")

fig, ax = plt.subplots(figsize=(8, 4.6))
order = top_share.sort_values(ascending=False)
colors = [C.crimson if v > 0.80 else C.navy for v in order.values]
ax.barh(range(len(order)), order.values, color=colors)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order.index, fontsize=7)
ax.invert_yaxis()
ax.axvline(0.80, color=C.gold, ls="--", lw=1.5, label="near-constant cutoff = 0.80")
ax.set_xlabel("Share of the single most frequent value")
ax.set_title("Near-constant Descriptor Screening on the Training Set")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
NEAR_CONST_CUTOFF = 0.80
const_like = sorted(top_share[top_share > NEAR_CONST_CUTOFF].index)
print(f"判定为准常量的描述符（{len(const_like)} 个）:", const_like)

cols_var = [c for c in X_full.columns if c not in const_like]
rf_var = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr[cols_var], y_tr)
print(f"剔除准常量特征后：{len(cols_var)} 维，测试集 R2 = {rf_var.score(X_te[cols_var], y_te):.4f}"
      f"（基准 {r2_full:.4f}）")

注意两件事：

1. **方差过滤是完全无监督的**——它从头到尾没有看过标签 $y$。因此它只适合用来扔掉“几乎恒定”的废列，**绝不能用来给特征排重要性**。看最后那行对照输出：`MolLogP` 的 Min-Max 方差小到可以忽略（因为 AqSolDB 里混进了 $\log P \approx -40$ 的极端离子，把量程拉得极宽，归一化后绝大多数分子都挤在一小段里），但它恰恰是下文 SHAP 分析中统治一切的头号特征。
2. 准常量特征对**树模型**的伤害其实有限（分裂时自然会被忽略），但对**线性模型、kNN、神经网络**则会实打实地注入噪声与数值不稳定。

### 🏋️ 练习 05a-1：方差过滤阈值的敏感性分析

把“最高频取值占比”的阈值 `NEAR_CONST_CUTOFF` 依次取 `0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50`：
1. 对每个阈值统计保留下来的特征数，并用随机森林（`n_estimators=200, random_state=0`）在训练集上拟合、在测试集上评估 $R^2$；
2. 画出“保留特征数 vs 测试集 $R^2$”的折线图，判断阈值降到多少时精度开始明显下滑。

In [ ]:
# 在下方填写你的代码：


---

## 4. 过滤法（二）：共线性剪枝

比准常量特征危害大得多的是**共线性**。在第二讲 `02c` 的置换重要性实验里我们已经见过这个现象：打乱 `MolWt` 时模型纹丝不动，因为 `HeavyAtomCount` 还在旁边提供了几乎相同的信息——**两个高度相关的特征会互相“打掩护”，让任何基于“删掉它会掉多少分”的重要性度量同时低估两者**。

诊断工具就是 **Pearson 相关系数矩阵** $|r_{ij}|$。剪枝策略是一个朴素的贪心算法：

> 按某个优先级遍历特征，一旦发现某列与**已保留**的某列相关系数 $|r| > 0.9$，就把它丢掉。

优先级怎么定？两种常见做法：(a) 按与标签 $y$ 的相关性从高到低（保留信息量大的）；(b) 按化学可解释性人工排序（保留实验同行看得懂的）。这里我们用 (a)。

In [ ]:
corr = X_tr[cols_var].corr().abs()

fig, ax = plt.subplots(figsize=(8.2, 7))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=0, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns, fontsize=7)
ax.set_title("Absolute Pearson Correlation Matrix of RDKit Descriptors")
plt.colorbar(im, ax=ax, shrink=0.75, label="|r|")
plt.show()

pairs = [
    (a, b, corr.loc[a, b])
    for i, a in enumerate(corr.columns) for b in corr.columns[i + 1:]
]
pairs.sort(key=lambda t: -t[2])
print("相关性最强的 10 对描述符：")
for a, b, r in pairs[:10]:
    print(f"  |r| = {r:.3f}   {a:<24s} <-> {b}")

In [ ]:
def prune_collinear(corr_mat, priority, threshold=0.90):
    """贪心共线性剪枝：按 priority 顺序遍历，丢掉与已保留列相关系数超过阈值的列。"""
    kept, dropped = [], {}
    for col in priority:
        clash = next((k for k in kept if corr_mat.loc[k, col] > threshold), None)
        if clash is None:
            kept.append(col)
        else:
            dropped[col] = (clash, corr_mat.loc[clash, col])
    return kept, dropped

# 优先级：与 logS 的绝对相关系数从高到低
target_corr = (X_tr[cols_var]
               .apply(lambda s: abs(np.corrcoef(s, y_tr)[0, 1]))
               .sort_values(ascending=False))
cols_pruned, dropped_map = prune_collinear(corr, list(target_corr.index), threshold=0.90)

print(f"共线性剪枝：{len(cols_var)} -> {len(cols_pruned)} 维")
print("\n被剔除的描述符及其替身：")
for col, (keeper, r) in dropped_map.items():
    print(f"  {col:<24s} 被 {keeper:<20s} 覆盖 (|r| = {r:.3f})")

rf_pruned = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr[cols_pruned], y_tr)
print(f"\n剪枝后测试集 R2 = {rf_pruned.score(X_te[cols_pruned], y_te):.4f}（基准 {r2_full:.4f}）")

---

## 5. 包裹法：递归特征消除（RFE / RFECV）

过滤法只看特征之间的统计关系，完全不知道最终模型的脾气。**递归特征消除（Recursive Feature Elimination, RFE）** 则反过来，把模型拉进循环：

1. 用**全部**特征训练一次模型，读出每个特征的内置重要性（树模型的 `feature_importances_`，线性模型的 $|\text{coef}|$）；
2. 扔掉重要性最低的 `step` 个特征；
3. 在剩下的特征上**重新训练**，重复第 2 步，直到只剩 `n_features_to_select` 个。

关键在于第 3 步的“重新训练”：每轮淘汰之后，幸存特征的重要性会被重新评估。这恰恰能部分化解共线性——`MolWt` 和 `HeavyAtomCount` 一开始互相分摊重要性，但只要其中一个先被淘汰，另一个的重要性立刻会在下一轮里反弹上来。

代价是**训练次数成倍增长**：29 个特征、`step=2` 意味着要训练约 10 次随机森林。如果还不知道该保留几个特征，`RFECV` 会在每个候选维度上再套一层交叉验证。

In [ ]:
N_KEEP = 10
rfe = RFE(
    estimator=RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1),
    n_features_to_select=N_KEEP,
    step=2,
)
rfe.fit(X_tr, y_tr)

cols_rfe = list(X_full.columns[rfe.support_])
rank = pd.Series(rfe.ranking_, index=X_full.columns).sort_values()
print(f"RFE 选出的 {N_KEEP} 个描述符：{cols_rfe}")
print("\n淘汰顺序（ranking_ = 1 表示入选，数字越大越早被淘汰）：")
print(rank.to_string())

rf_rfe = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr[cols_rfe], y_tr)
print(f"\nRFE 10 维测试集 R2 = {rf_rfe.score(X_te[cols_rfe], y_te):.4f}（基准 {r2_full:.4f}）")

### 🏋️ 练习 05a-2：用 `RFECV` 自动确定该保留几个特征

手动把 `N_KEEP` 定成 10 是拍脑袋的。`sklearn.feature_selection.RFECV` 会在每一个候选特征数上做交叉验证，自动挑出交叉验证得分最高的维度。

1. 用 `RFECV(estimator=RandomForestRegressor(n_estimators=60, random_state=0, n_jobs=-1), step=2, cv=KFold(3, shuffle=True, random_state=0), scoring="r2", min_features_to_select=3)` 在 `(X_tr, y_tr)` 上拟合（约需 1–3 分钟）；
2. 打印 `rfecv.n_features_` 与入选特征名；
3. 用 `rfecv.cv_results_["mean_test_score"]` 画出“特征数 vs 交叉验证 $R^2$”曲线，并在最优点上标注一条竖线。

> **提示**：`cv_results_["mean_test_score"]` 的第 $k$ 个元素对应 `min_features_to_select + k * step` 个特征。

In [ ]:
# 在下方填写你的代码：


---

## 6. 三条路线横评：29 维 → 10 维，精度几乎不掉

把前面三种策略放到同一把尺子下，用**五折交叉验证**（而不是单次划分）比较，以免被一次随机划分的运气骗了：

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=0)
strategies = {
    "1_all_descriptors":      list(X_full.columns),
    "2_variance_filter":      cols_var,
    "3_variance_plus_prune":  cols_pruned,
    "4_rfe_wrapper":          cols_rfe,
}

report = []
for name, cols in strategies.items():
    scores = cross_val_score(
        RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1),
        X_full[cols], y, cv=cv, scoring="r2",
    )
    report.append({
        "strategy":   name,
        "n_features": len(cols),
        "cv_r2_mean": round(scores.mean(), 4),
        "cv_r2_std":  round(scores.std(), 4),
    })

report = pd.DataFrame(report)
print(report.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.errorbar(report["n_features"], report["cv_r2_mean"], yerr=report["cv_r2_std"],
            fmt="o-", capsize=4, lw=1.8, ms=8, color=C.navy)
for _, r in report.iterrows():
    ax.annotate(r["strategy"][0], (r["n_features"], r["cv_r2_mean"]),
                textcoords="offset points", xytext=(6, 7), fontsize=11)
ax.set_xlabel("Number of descriptors kept")
ax.set_ylabel("5-fold CV R2")
ax.set_title("Feature Selection: Fewer Descriptors, Same Accuracy")
plt.tight_layout()
plt.show()

**结论非常清楚**：从 29 维压到 10 维，交叉验证 $R^2$ 的变化完全落在折间标准差之内——**三分之二的描述符是可以安全删掉的冗余**。换来的是更短的训练时间、更稳的小样本表现，以及一张能在组会上逐行讲清楚的特征表。

---

## 7. 防坑：特征选择必须放进 Pipeline

这是一个**极其常见、极其隐蔽**的数据泄漏陷阱，很多已发表的 QSAR 论文都栽在这里：

> ❌ **错误做法**：在**整个数据集**上跑一次 RFE 选出 10 个特征，然后再对这 10 个特征做交叉验证报告性能。

错在哪？RFE 在选特征时**看过了全部样本的标签 $y$**，包括后来被划进验证折的那些。于是“哪些特征有用”这个决定本身就泄漏了验证集信息，交叉验证分数会被系统性地抬高。

正确做法是把特征选择器**当成模型的一部分**塞进 `Pipeline`，让它在每一折的训练子集上独立地重新选一次：

In [ ]:
# 错误：先在全量数据上选特征，再交叉验证
leaky_selector = RFE(RandomForestRegressor(n_estimators=60, random_state=0, n_jobs=-1),
                     n_features_to_select=8, step=3).fit(X_full, y)
leaky_cols = X_full.columns[leaky_selector.support_]
leaky_score = cross_val_score(
    RandomForestRegressor(n_estimators=150, random_state=0, n_jobs=-1),
    X_full[leaky_cols], y, cv=cv, scoring="r2",
).mean()

# 正确：特征选择进 Pipeline，每折独立重选
honest_pipe = Pipeline([
    ("select", RFE(RandomForestRegressor(n_estimators=60, random_state=0, n_jobs=-1),
                   n_features_to_select=8, step=3)),
    ("model",  RandomForestRegressor(n_estimators=150, random_state=0, n_jobs=-1)),
])
honest_score = cross_val_score(honest_pipe, X_full, y, cv=cv, scoring="r2").mean()

print(f"[错误] 全量选特征后再 CV   : R2 = {leaky_score:.4f}")
print(f"[正确] 选特征嵌入 Pipeline : R2 = {honest_score:.4f}")
print(f"\n乐观偏差 = {leaky_score - honest_score:+.4f}")

在本例中（3,000 个样本、29 个特征、特征与标签关系极强），两者的差距小到被随机波动淹没——偏差甚至可能是个负数。这不代表泄漏无害，而是说明**在样本远多于特征、信号又很强时，"选哪些特征"这个决定本身几乎不含可泄漏的信息**。请记住这个规律：**样本越少、候选特征越多，这个偏差就越致命**。在一个 40 个实测样本、1,800 维 mordred 描述符的课题里，“先全量选特征再 CV” 能把一个纯噪声模型的 $R^2$ 吹到 0.8 以上。

> **📌 一句话准则**：凡是“看过标签 $y$”的预处理步骤（特征选择、目标编码、过采样），都必须放进 `Pipeline`；只看 $X$ 的步骤（`StandardScaler`、`VarianceThreshold`）虽然泄漏风险小，放进去也更稳妥。

---

## 8. 从全局重要性到单分子归因：Shapley 值

到这里，我们能回答的还只是“**整体上**哪些描述符重要”。但当你合成出一个新分子、模型预测 $\log S = -6.4$ 时，你真正想知道的是：

> 这 $-6.4$ 里，有多少是因为它 $\log P$ 高？有多少是因为它芳环多、刚性大？

SHAP 给出的答案来自 1953 年 Lloyd Shapley 的**合作博弈论**。把一次预测看成一场“合作”：$M$ 个特征是 $M$ 位玩家，模型输出是他们共同创造的“收益”。Shapley 值要解决的就是**如何公平地把总收益分配给每位玩家**。

对特征 $i$，其 Shapley 值定义为它在**所有可能的加入顺序**中带来的边际贡献的平均：

$$\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!\,(M - |S| - 1)!}{M!} \Big[ v(S \cup \{i\}) - v(S) \Big]$$

其中 $F$ 是全部特征集合，$S$ 是不含特征 $i$ 的任一子集，$v(S)$ 是“只用 $S$ 中的特征”时模型对该样本的预测值。前面那个组合系数正是“在随机排列中，恰好 $S$ 里的玩家排在 $i$ 之前”的概率。

Shapley 值是**唯一**同时满足以下四条公理的分配方案：

| 公理 | 含义 |
|---|---|
| **有效性（Efficiency）** | $\sum_i \phi_i = v(F) - v(\varnothing)$，即所有贡献之和恰好等于“该样本的预测值 − 全体平均预测值” |
| **对称性（Symmetry）** | 两个作用完全相同的特征，分到的贡献必须相等 |
| **虚拟性（Dummy）** | 对任何子集都不产生影响的特征，贡献为 0 |
| **可加性（Additivity）** | 两个模型相加，其 Shapley 值也相加（这是 SHAP 能处理随机森林这种集成模型的关键） |

**有效性公理**正是 SHAP 最漂亮的地方：它保证了一个严格的加法分解

$$\hat{y}(x) = \underbrace{\mathbb{E}[\hat{y}]}_{\text{base value}} + \sum_{i=1}^{M} \phi_i(x)$$

每个分子的预测值都能被**不多不少**地拆成“数据集平均值 + 各描述符的推拉贡献”。

### 8.1 手写精确 Shapley 值：用 16 个重训模型验证公理

公式里的 $v(S)$（“只用 $S$ 中特征的模型”）严格来说需要**对每一个子集重新训练一个模型**。$M$ 个特征就是 $2^M$ 个模型——这正是 Shapley 值的组合爆炸。但如果我们只取 4 个描述符，$2^4 = 16$ 个随机森林在几秒内就能训完，足以让我们**完全按定义**把 Shapley 值算一遍，并亲眼验证有效性公理。

In [ ]:
F4 = ["MolLogP", "MolWt", "TPSA", "NumAromaticRings"]
X4_tr, X4_te = X_tr[F4], X_te[F4]
BASE_VALUE = y_tr.mean()          # v(空集)：不用任何特征时的最优预测 = 训练集均值

# 为全部 2^4 = 16 个特征子集各训练一个随机森林
sub_models = {}
for r in range(len(F4) + 1):
    for S in itertools.combinations(F4, r):
        sub_models[S] = None if not S else RandomForestRegressor(
            n_estimators=60, random_state=0, n_jobs=-1
        ).fit(X4_tr[list(S)], y_tr)

print(f"已训练 {len(sub_models)} 个子集模型（含空集）")

def v(S, x):
    """价值函数 v(S)：只用 S 中的特征时，模型对样本 x 的预测值。"""
    S = tuple(f for f in F4 if f in S)        # 固定顺序，便于查表
    if not S:
        return BASE_VALUE
    return sub_models[S].predict(pd.DataFrame([x[list(S)]]))[0]

def exact_shapley(x):
    """严格按定义枚举全部子集，计算精确 Shapley 值。"""
    M_ = len(F4)
    phi = {}
    for f in F4:
        others = [g for g in F4 if g != f]
        total = 0.0
        for r in range(len(others) + 1):
            for S in itertools.combinations(others, r):
                w = math.factorial(len(S)) * math.factorial(M_ - len(S) - 1) / math.factorial(M_)
                total += w * (v(set(S) | {f}, x) - v(set(S), x))
        phi[f] = total
    return phi

In [ ]:
x0 = X4_te.iloc[0]
phi0 = exact_shapley(x0)
pred_full = v(F4, x0)

print("样本的 4 个描述符取值：")
print(x0.round(3).to_string())
print("\n精确 Shapley 值（单位：log S）：")
for f, val in sorted(phi0.items(), key=lambda kv: -abs(kv[1])):
    print(f"  phi({f:<18s}) = {val:+.4f}")

print("\n有效性公理检验：")
print(f"  base value E[y]     = {BASE_VALUE:.6f}")
print(f"  + sum(phi_i)        = {sum(phi0.values()):+.6f}")
print(f"  = {BASE_VALUE + sum(phi0.values()):.6f}")
print(f"  全特征模型预测 v(F)  = {pred_full:.6f}")
print(f"  实测 logS           = {y_te[0]:.4f}")
print(f"\n  残差 = {abs(BASE_VALUE + sum(phi0.values()) - pred_full):.2e}  -> 有效性公理精确成立")

残差是机器精度量级——**有效性公理严格成立**，不是近似。

但这种“每个子集重训一个模型”的做法有两个硬伤：(1) $2^M$ 的组合爆炸，29 个特征就是 5 亿个模型；(2) 它衡量的是“这个特征值不值得被收集”，而非“这个已训练好的模型是怎么用它的”。

实践中用的是 **TreeSHAP** 算法：它不重训模型，而是把“缺失特征”处理成“在树里沿着两个分支按样本比例加权下行”，从而在 $O(T L D^2)$ 时间内（$T$ 棵树、$L$ 个叶子、$D$ 层深）**精确**算出树模型的 Shapley 值——这正是 `shap.TreeExplainer` 能在几秒内处理几百棵树的原因。

---

## 9. `shap.TreeExplainer`：全局归因

下面在第 5 节 RFE 选出的 10 维特征上训练随机森林，然后用 TreeSHAP 解释测试集中的 200 个分子。

In [ ]:
try:
    import shap
except ImportError:
    print("正在安装 shap ……")
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "shap"], check=True)
    import shap

print("shap 版本:", shap.__version__)

model_shap = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr[cols_rfe], y_tr)

explainer = shap.TreeExplainer(model_shap)
X_explain = X_te[cols_rfe].iloc[:200]
shap_values = explainer.shap_values(X_explain)
base_value = float(np.ravel(explainer.expected_value)[0])

print(f"SHAP 值矩阵形状: {shap_values.shape}   (200 个分子 x {len(cols_rfe)} 个特征)")
print(f"base value E[y] = {base_value:.4f}")

recon = shap_values.sum(axis=1) + base_value
print(f"可加性检验 max|重构值 - 模型预测| = "
      f"{np.abs(recon - model_shap.predict(X_explain)).max():.2e}")

In [ ]:
# shap 内部会调用 plt.tight_layout()，与课程样式的 constrained_layout 冲突，
# 因此所有 shap 画图都放进一个临时关掉 constrained_layout 的上下文里。
with plt.rc_context({"figure.constrained_layout.use": False}):
    shap.summary_plot(shap_values, X_explain, show=False, plot_size=(8, 5))
    plt.title("SHAP Beeswarm: How Each Descriptor Pushes logS", fontsize=12)
    plt.show()

**蜂群图（beeswarm）怎么读**，这是 SHAP 最高信息密度的一张图：

- **纵轴**：特征按 $\overline{|\phi_i|}$（平均绝对贡献）从上到下排序——这是 SHAP 的全局重要性；
- **横轴**：该特征对**单个分子**预测值的推拉量 $\phi_i$（单位就是 $\log S$）。在 0 右侧 = 把预测推向“更易溶”，左侧 = 推向“更难溶”；
- **颜色**：该分子在这个特征上的**取值高低**（红高蓝低）。

于是第一行 `MolLogP` 呈现出教科书般的图案：**红点（高 $\log P$）全部堆在左边，蓝点（低 $\log P$）全部堆在右边**——疏水性越强，模型就越往“难溶”方向推，而且最极端的分子能被单独这一个特征推动 3–4 个对数单位。这正是 Yalkowsky 方程里 $-\log P$ 那一项的机器学习版本。

In [ ]:
mean_abs = pd.Series(np.abs(shap_values).mean(axis=0), index=cols_rfe).sort_values()

fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.barh(mean_abs.index, mean_abs.values, color=C.navy)
ax.set_xlabel("mean(|SHAP value|)   [log S units]")
ax.set_title("Global Feature Importance from SHAP")
for i, v_ in enumerate(mean_abs.values):
    ax.text(v_ + 0.01, i, f"{v_:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

print(mean_abs.sort_values(ascending=False).round(4).to_string())

---

## 10. 局部归因：为什么模型判这一个分子难溶？

这才是 SHAP 相对于 MDI / 置换重要性的**本质优势**：后两者只有一个全局数字，而 SHAP 对**每一个分子**都给出一套独立的归因。

下面我们挑出测试集中模型预测最难溶的分子，画出它的**瀑布图（waterfall plot）**——从数据集平均 $\mathbb{E}[\hat{y}]$ 出发，看各个描述符如何一步步把预测推到最终值。

In [ ]:
pred_explain = model_shap.predict(X_explain)
idx_hard = int(np.argmin(pred_explain))        # 预测最难溶的分子

orig_idx = X_explain.index[idx_hard]
smi_hard = sub.loc[orig_idx, "SMILES"]
name_hard = sub.loc[orig_idx, "Name"]

print(f"分子名称  : {name_hard}")
print(f"SMILES    : {smi_hard}")
print(f"实测 logS : {sub.loc[orig_idx, 'Solubility']:.3f}")
print(f"预测 logS : {pred_explain[idx_hard]:.3f}")

mol_hard = Chem.MolFromSmiles(smi_hard)
Draw.MolToImage(mol_hard, size=(340, 260))

In [ ]:
explanation = shap.Explanation(
    values=shap_values,
    base_values=np.repeat(base_value, len(X_explain)),
    data=X_explain.values,
    feature_names=cols_rfe,
)

with plt.rc_context({"figure.constrained_layout.use": False}):
    shap.plots.waterfall(explanation[idx_hard], max_display=10, show=False)
    plt.title("Waterfall: from E[y] to this molecule's predicted logS", fontsize=11)
    plt.show()

瀑布图从底部的 $\mathbb{E}[\hat{y}]$ 起步，每一条红/蓝色带把预测往上/往下推一段，最终严格落在模型预测值 $f(x)$ 上——**这就是有效性公理的可视化**。

把这张图翻译成化学语言，就是一句可以直接写进论文讨论部分的话：*“模型判定该分子难溶，主要源于其极高的 `MolLogP` 与较大的 `BertzCT` 结构复杂度，而较高的 `TPSA` 只能部分抵消这一趋势。”*

### 🏋️ 练习 05a-3：对比三个并苯类有机半导体分子的 SHAP 归因

用上面训练好的 `model_shap`（输入特征为 `cols_rfe`）解释下面三个分子：

- `Naphthalene`（萘）: `"c1ccc2ccccc2c1"`
- `Anthracene`（蒽）: `"c1ccc2cc3ccccc3cc2c1"`
- `Pentacene`（并五苯）: `"c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1"`

1. 为这三个分子计算 `cols_rfe` 中的全部描述符（`cols_rfe` 中的名字与 `Descriptors` / `rdMolDescriptors` 中的计算函数同名，可以用一个字典 `ALL_DESC = {**{c: getattr(Descriptors, c) for c in BASE_COLS}, **EXTRA_DESC}` 统一取到）；
2. 调用 `explainer.shap_values()` 得到它们的 SHAP 值，整理成一张 DataFrame（行 = 分子，列 = 特征，另加 `base_value` 与 `预测 logS` 两列）；
3. 用分组柱状图并排对比三者，回答：**从萘到并五苯，预测值的下降主要由哪个描述符贡献？**

In [ ]:
# 在下方填写你的代码：


---

## 11. SHAP vs MDI vs 置换重要性：共线特征下的三种表现

第二讲 `02c` 介绍过两种全局重要性，加上 SHAP 一共三种。它们的差别在**共线特征**上暴露得最充分：

In [ ]:
# 刻意在高度共线的特征对上做对比：MolWt 与 HeavyAtomCount 同时在场
COLLINEAR_SET = ["MolLogP", "MolWt", "HeavyAtomCount", "MolMR", "TPSA", "NumAromaticRings"]
m_col = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X_tr[COLLINEAR_SET], y_tr)

mdi = pd.Series(m_col.feature_importances_, index=COLLINEAR_SET)

perm = permutation_importance(m_col, X_te[COLLINEAR_SET], y_te,
                              n_repeats=10, random_state=0, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=COLLINEAR_SET)

sv_col = shap.TreeExplainer(m_col).shap_values(X_te[COLLINEAR_SET].iloc[:200])
shap_imp = pd.Series(np.abs(sv_col).mean(axis=0), index=COLLINEAR_SET)

comp = pd.DataFrame({
    "MDI_tree_split": mdi / mdi.sum(),
    "Permutation":    perm_imp / perm_imp.sum(),
    "SHAP_mean_abs":  shap_imp / shap_imp.sum(),
})
print("（每列已归一化为占比，便于横向对比）")
print(comp.round(3).to_string())

comp.plot(kind="bar", figsize=(8.5, 4.2), rot=30, width=0.78,
          color=[C.navy, C.gold, C.crimson])
plt.ylabel("Normalized importance share")
plt.title("Three Importance Metrics under Strong Collinearity")
plt.tight_layout()
plt.show()

print(f"\n参考：|r(MolWt, HeavyAtomCount)| = "
      f"{abs(np.corrcoef(X_tr['MolWt'], X_tr['HeavyAtomCount'])[0, 1]):.3f}")

三者的区别可以总结为：

| 度量 | 算在哪个数据集上 | 对共线特征的行为 | 主要局限 |
|---|---|---|---|
| **MDI（树分裂重要性）** | 训练集（且只是训练过程的副产物） | 共线特征**平分**重要性，各自被低估 | 系统性偏爱高基数（取值多）的连续特征；在训练集上算，会把过拟合也算成“重要” |
| **置换重要性（Permutation）** | 测试集 | 共线特征**互相打掩护**，可能双双接近 0 | 打乱单列会制造化学上不可能存在的分子（如 $\log P = 8$ 却 `MolWt` $= 60$） |
| **SHAP** | 任意指定样本集 | 共线特征按模型实际使用情况**分摊**贡献，不会双双归零 | 计算最贵；TreeSHAP 的条件期望处理同样受相关性影响 |

> **📌 实践建议**：先用共线性剪枝把高度相关的特征收拾干净（第 4 节），再用 SHAP 做归因——这样三种度量的结论会高度一致，报告出来也最可信。

---

## 小结

| 核心概念 | 方法与实现 | 在材料/化学数据上的意义 |
|---|---|---|
| **过滤法：方差 / 准常量** | `VarianceThreshold`；或统计最高频取值占比 | 剔除 `NumRadicalElectrons` 这类在有机分子库里恒为 0 的废列；**必须先归一化**，否则只是在比量纲 |
| **过滤法：共线性剪枝** | $\lvert r_{ij}\rvert > 0.9$ 贪心剪枝 | `MolWt` / `HeavyAtomCount` / `MolMR` / `LabuteASA` / `Chi0v` 本质同义，保留一个即可，维度直接腰斩而精度不变 |
| **包裹法：RFE / RFECV** | 反复训练 + 淘汰最弱特征 | 最贴合最终模型；29 → 10 维，五折 CV $R^2$ 落在折间标准差内 |
| **选择性数据泄漏** | 特征选择必须进 `Pipeline` | 在全量数据上选特征再 CV 会系统性高估性能，小样本高维时尤其致命 |
| **Shapley 值四公理** | $\phi_i = \sum_S w_S\,[\,v(S\cup\{i\}) - v(S)\,]$ | 有效性公理保证 $\hat{y} = \mathbb{E}[\hat{y}] + \sum_i \phi_i$，预测值被不多不少地分解 |
| **TreeSHAP 全局归因** | `shap.TreeExplainer` + 蜂群图 | `MolLogP` 红点左、蓝点右的图案，就是 Yalkowsky 方程 $-\log P$ 项的机器学习版本 |
| **TreeSHAP 局部归因** | 瀑布图 `shap.plots.waterfall` | 对**单个新合成分子**给出“因为疏水性太强、结构太刚性所以难溶”的可写进论文的归因句 |

**下一节预告（`05b_conformers_and_pubchem.ipynb`）**：本节所有描述符都是从 2D 拓扑图上算出来的。但 Yalkowsky 方程里还有一项 $-0.01(T_m - 25)$——**熔点**，它来自三维晶体堆积，任何 2D 描述符都算不出来。下一节我们先用 RDKit 的 ETKDG + MMFF 把分子“站起来”变成三维构象系综，再用 PubChem PUG-REST API 从数据库里把真实的实验熔点抓回来，亲手补上这块从第一讲就欠下的拼图。